In [1]:
import zipfile
import numpy as np
import pandas as pd

import networkx as nx

from code_utils.utils_basic import PROJECTED_CRS, PROJECT_PATH

# 1 Convert edge list to graph

In [2]:
def print_df_info(df):
    print('\nData info:',
        '\nData shape:', df.shape,
        '\nData columns:', df.columns.tolist(),
        '\nNA values:', df.isna().sum().sum()
    )
# ============================================================
def print_graph_info(G, weight=None):
    """
    Print key statistics of a NetworkX graph.

    Parameters:
    -----------
    graph : networkx.Graph
        The input graph.
    weight : str or None, optional (default='distance')
        The edge attribute to use as weight. If None, unweighted distances are used.
    """
    print('\nGraph brief statistics:',
        '\nDirected:', nx.is_directed(G),
        '\nNumber of nodes:', G.number_of_nodes(),
        '\nNumber of edges:', G.number_of_edges(),
        # '\nNetwork diameter:', nx.diameter(G, weight=weight),
        # '\nAverage shortest distance:', nx.average_shortest_path_length(G, weight=weight),
        '\nNetwork density:', nx.density(G),
        '\nNumber of self-loops:', nx.number_of_selfloops(G),
        '\nNumber of isolated nodes:', len(list(nx.isolates(G)))
    )

    # Total edge weight (if applicable)
    if weight is not None:
        weight_li = list(nx.get_edge_attributes(G, weight).values())
        print('\nEdge weight info:',
            '\n\tEdge weight attribute:', weight,
            f'\n\tMin = {float(np.min(weight_li)):.2f}, '
                f'Max = {float(np.max(weight_li)):.2f}, '
            f'\n\tMean = {float(np.mean(weight_li)):.2f}, '
                f'Median = {float(np.median(weight_li)):.2f}'
            f'\n\tStd = {float(np.std(weight_li)):.2f}',
            f'\n\tSum: {sum(weight_li):.2f}')

    # node attribute
    node_df = pd.DataFrame.from_dict(dict(G.nodes(data=True)), orient='index')
    # edge attribute
    edge_df = nx.to_pandas_edgelist(G)

    print('\nNode info:',
        '\n\tData shape:', node_df.shape,
        '\n\tColumns:', node_df.columns.tolist(),
        '\n\tNA values:', node_df.isna().sum().to_dict(),
        '\nEdge info:',
        '\n\tData shape:', edge_df.shape,
        '\n\tColumns:', edge_df.columns.tolist(),
        '\n\tNA values:', edge_df.isna().sum().to_dict(),
    )

    return None
# ===========================================================
def load_csv_from_zip(zip_path, csv_path, **kwargs):
    """
    Load a CSV file from a ZIP archive into a pandas DataFrame.
    """
    with zipfile.ZipFile(zip_path, 'r') as zf:
        with zf.open(csv_path) as f:
            df = pd.read_csv(f, **kwargs)
    return df

#### Load edge list

In [3]:
zip_path = PROJECT_PATH / 'data/bus_network/bus_graph_20221020.zip'

# Load edge list for space L
edge_list_l = load_csv_from_zip(
    zip_path, csv_path = 'edge_list_space_l.csv',
    index_col = None,
    dtype = {
        'source'   : str,
        'target'   : str,
        'distance' : float,
        'serv_num' : int,
        'serv_li'  : str
    })

# Ensure edge distance is positive
# edge_list = edge_list[edge_list['distance'] > 0.]
print_df_info(edge_list_l)


# Load edge list for space P
edge_list_p = load_csv_from_zip(
    zip_path, csv_path = 'edge_list_space_p.csv',
    index_col = None,
    dtype = {
        'source'   : str,
        'target'   : str,
        'dist'     : float,
        'serv_num' : int,
        'serv_li'  : str})

# Ensure edge distance is positive
# edge_list = edge_list[edge_list['distance'] > 0.]
print_df_info(edge_list_p)


Data info: 
Data shape: (7396, 5) 
Data columns: ['source', 'target', 'distance', 'serv_no', 'serv_li'] 
NA values: 0

Data info: 
Data shape: (408903, 5) 
Data columns: ['source', 'target', 'distance', 'serv_no', 'serv_li'] 
NA values: 0


#### Create graph based on edge list

In [4]:
# L space
G = nx.from_pandas_edgelist(
    edge_list_l,
    source = 'source', target = 'target',
    create_using = nx.DiGraph,
    edge_attr = True)
# Drop self-loops
G.remove_edges_from(nx.selfloop_edges(G))
# Drop isolated nodes
G.remove_nodes_from(list(nx.isolates(G)))

print_graph_info(G, weight='distance')
# Save graph
# nx.write_graphml(G, PROJECT_PATH / 'data/bus_network/graph_20221020/graph_space_l.graphml')


# P space
G = nx.from_pandas_edgelist(
    edge_list_p,
    source = 'source', target = 'target',
    create_using = nx.DiGraph,
    edge_attr = True)
# Drop self-loops
G.remove_edges_from(nx.selfloop_edges(G))
# Drop isolated nodes
G.remove_nodes_from(list(nx.isolates(G)))

print_graph_info(G, weight='distance')
# Save graph
# nx.write_graphml(G, PROJECT_PATH / 'data/bus_network/graph_20221020/graph_space_p.graphml')


Graph brief statistics: 
Directed: True 
Number of nodes: 5075 
Number of edges: 7385 
Network density: 0.0002867899908933984 
Number of self-loops: 0 
Number of isolated nodes: 0

Edge weight info: 
	Edge weight attribute: distance 
	Min = 0.00, Max = 23.10, 
	Mean = 0.66, Median = 0.40
	Std = 1.64 
	Sum: 4865.80

Node info: 
	Data shape: (0, 0) 
	Columns: [] 
	NA values: {} 
Edge info: 
	Data shape: (7385, 5) 
	Columns: ['source', 'target', 'serv_li', 'serv_no', 'distance'] 
	NA values: {'source': 0, 'target': 0, 'serv_li': 0, 'serv_no': 0, 'distance': 0}

Graph brief statistics: 
Directed: True 
Number of nodes: 5075 
Number of edges: 408835 
Network density: 0.01587674826362932 
Number of self-loops: 0 
Number of isolated nodes: 0

Edge weight info: 
	Edge weight attribute: distance 
	Min = 0.00, Max = 72.90, 
	Mean = 10.51, Median = 8.60
	Std = 8.45 
	Sum: 4298848.30

Node info: 
	Data shape: (0, 0) 
	Columns: [] 
	NA values: {} 
Edge info: 
	Data shape: (408835, 5) 
	Columns: ['